In [7]:
from pathlib import Path
import pandas as pd

ROOT = Path("..")
RAW = ROOT / "data" / "raw" / "twcs.csv"

df_head = pd.read_csv(RAW, nrows=5)
df_head

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2.0,3
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1.0,4
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3.0,5
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4.0,6


In [9]:
# Apple support executive replies
chunks = []

for chunk in pd.read_csv(RAW, chunksize=50_000):
    chunks.append(chunk[chunk["author_id"] == "AppleSupport"])

df_apple_agent = pd.concat(chunks, ignore_index=True)

print(df_apple_agent.shape)
df_apple_agent.head()

(106860, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,696,AppleSupport,False,Tue Oct 31 22:27:49 +0000 2017,@115854 We're here for you. Which version of t...,697,698.0
1,699,AppleSupport,False,Tue Oct 31 22:36:27 +0000 2017,@115854 Lets take a closer look into this issu...,NaN,697.0
2,701,AppleSupport,False,Tue Oct 31 22:26:49 +0000 2017,@115855 Let's go to DM for the next steps. DM ...,NaN,702.0
3,703,AppleSupport,False,Tue Oct 31 22:09:52 +0000 2017,@115855 Any steps tried since it started last ...,702,704.0
4,705,AppleSupport,False,Tue Oct 31 21:57:00 +0000 2017,@115855 That's great it has iOS 11.1 as we can...,"706,704",707.0


In [11]:
# Apple customers queries or tweets
chunks = []

for chunk in pd.read_csv(RAW, chunksize=50_000):
    is_agent = chunk["author_id"] == "AppleSupport"
    is_customer = chunk["text"].str.contains("@AppleSupport", case=False, na=False)
    chunks.append(chunk[is_agent | is_customer])

df_apple = pd.concat(chunks, ignore_index=True)

print(df_apple.shape)
df_apple["inbound"].value_counts()

(204772, 7)


inbound
False    106876
True      97896
Name: count, dtype: int64

In [12]:
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

In [13]:
df_apple_agent.to_parquet(
    OUT / "apple_agent_tweets.parquet",
    index=False
)

# df_apple.to_parquet(OUT / "apple_tweets.parquet", index=False)

In [14]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(
    Path("..") / "data" / "processed" / "apple_agent_tweets.parquet"
)

print(df.shape)
df.head()

(106860, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,696,AppleSupport,False,Tue Oct 31 22:27:49 +0000 2017,@115854 We're here for you. Which version of t...,697,698.0
1,699,AppleSupport,False,Tue Oct 31 22:36:27 +0000 2017,@115854 Lets take a closer look into this issu...,NaN,697.0
2,701,AppleSupport,False,Tue Oct 31 22:26:49 +0000 2017,@115855 Let's go to DM for the next steps. DM ...,NaN,702.0
3,703,AppleSupport,False,Tue Oct 31 22:09:52 +0000 2017,@115855 Any steps tried since it started last ...,702,704.0
4,705,AppleSupport,False,Tue Oct 31 21:57:00 +0000 2017,@115855 That's great it has iOS 11.1 as we can...,"706,704",707.0


In [15]:
# rows that look non-ASCII / CJK
non_ascii = df_apple_agent["text"].str.contains(r"[^\x00-\x7F]", na=False)

print("non_ascii", non_ascii.sum(), "of", len(df_apple_agent))

non_ascii 13340 of 106860


In [17]:
# keep tweets that are mostly ASCII (English-ish). Drops JP/CN/KR etc.
ascii_ratio = df_apple_agent["text"].fillna("").map(
    lambda t: sum(ch.isascii() for ch in t) / max(len(t), 1)
)

df_apple_en = df_apple_agent[ascii_ratio >= 0.9].copy()

print(df_apple_en.shape)
df_apple_en.head()

(106860, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,696,AppleSupport,False,Tue Oct 31 22:27:49 +0000 2017,@115854 We're here for you. Which version of t...,697,698.0
1,699,AppleSupport,False,Tue Oct 31 22:36:27 +0000 2017,@115854 Lets take a closer look into this issu...,NaN,697.0
2,701,AppleSupport,False,Tue Oct 31 22:26:49 +0000 2017,@115855 Let's go to DM for the next steps. DM ...,NaN,702.0
3,703,AppleSupport,False,Tue Oct 31 22:09:52 +0000 2017,@115855 Any steps tried since it started last ...,702,704.0
4,705,AppleSupport,False,Tue Oct 31 21:57:00 +0000 2017,@115855 That's great it has iOS 11.1 as we can...,"706,704",707.0


In [18]:
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

# working copy
df_apple_en.to_parquet(
    OUT / "apple_agent_en.parquet",
    index=False
)

# human-readable copy
df_apple_en.to_csv(
    OUT / "apple_agent_en.csv",
    index=False
)

print(df_apple_en.shape)

(106860, 7)


In [19]:
from pathlib import Path
import pandas as pd

ROOT = Path("..")
PROC = ROOT / "data" / "processed"
EDA = ROOT / "data" / "eda_results"
EDA.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROC / "apple_agent_en.parquet")
# if parquet missing:
# df = pd.read_csv(PROC / "apple_agent_en.csv")

print(df.shape)
df.head(3)

(106860, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,696,AppleSupport,False,Tue Oct 31 22:27:49 +0000 2017,@115854 We're here for you. Which version of t...,697,698.0
1,699,AppleSupport,False,Tue Oct 31 22:36:27 +0000 2017,@115854 Lets take a closer look into this issu...,NaN,697.0
2,701,AppleSupport,False,Tue Oct 31 22:26:49 +0000 2017,@115855 Let's go to DM for the next steps. DM ...,NaN,702.0


In [20]:
print(df.dtypes)

tweet_id                     int64
author_id                      str
inbound                       bool
created_at                     str
text                           str
response_tweet_id              str
in_response_to_tweet_id    float64
dtype: object


In [21]:
print("nulls:\n", df.isna())

nulls:
         tweet_id  author_id  inbound  created_at   text  response_tweet_id  \
0          False      False    False       False  False              False   
1          False      False    False       False  False               True   
2          False      False    False       False  False               True   
3          False      False    False       False  False              False   
4          False      False    False       False  False              False   
...          ...        ...      ...         ...    ...                ...   
106855     False      False    False       False  False              False   
106856     False      False    False       False  False               True   
106857     False      False    False       False  False               True   
106858     False      False    False       False  False               True   
106859     False      False    False       False  False               True   

        in_response_to_tweet_id  
0                    

In [22]:
print("nulls sum: \n", df.isna().sum())

nulls sum: 
 tweet_id                       0
author_id                      0
inbound                        0
created_at                     0
text                           0
response_tweet_id          75296
in_response_to_tweet_id      141
dtype: int64


In [23]:
print("before", len(df))

df_clean = (
    df.dropna(subset=["tweet_id", "author_id", "text"])   # real nulls only
      .drop_duplicates(subset=["tweet_id"])               # same tweet twice
      .loc[lambda x: x["text"].astype(str).str.strip().ne("")]
      .loc[lambda x: x["author_id"].astype(str).str.strip().ne("")]
      .copy()
)

print("after", len(df_clean))
print(df_clean.isna().sum())

before 106860
after 106860
tweet_id                       0
author_id                      0
inbound                        0
created_at                     0
text                           0
response_tweet_id          75296
in_response_to_tweet_id      141
dtype: int64


In [24]:
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["text"])
print("exact-text dups removed", n_before - len(df_clean))

exact-text dups removed 90


In [25]:
EDA = ROOT / "data" / "eda_results"
EDA.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(EDA / "apple_agent_en_clean.parquet", index=False)
df_clean.to_csv(EDA / "apple_agent_en_clean.csv", index=False)